# Restaurant Info & Order Tracking Agent with Guardrails

Run the cells from top to bottom to follow the agent flow.

### 1. Import dependencies and initialize the API clients.

In [ ]:
from dotenv import load_dotenv
from openai import OpenAI
import csv
import gradio as gr
import json
import os

load_dotenv(override=True)

openai = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

gemini_client = OpenAI(
    api_key=os.getenv("GOOGLE_API_KEY"),
    base_url="https://generativelanguage.googleapis.com/v1beta/openai/"
)


### 2. Load the restaurant knowledge base and assistant/evaluator rules.

In [ ]:
def read_file(filename):
    with open(filename, encoding="utf-8") as file:
        return file.read()

restaurant_info = read_file("restaurant.md")
assistant_rules = read_file("assistant_rules.md")
evaluator_rules = read_file("evaluator_rules.md")

system_prompt = f"""
You are the customer assistant for Desi Fork & Flame.

# Restaurant Information

{restaurant_info}

# Assistant Rules

{assistant_rules}
"""


### 3. Define the order lookup functions backed by the CSV mock database.

In [ ]:
def get_order(order_id):
    with open("restaurant_orders.csv", newline="", encoding="utf-8") as file:
        for row in csv.DictReader(file):
            if int(row["order_id"]) == int(order_id):
                return row

    return {"error": f"Order {order_id} was not found."}


def get_orders_by_customer(email):
    with open("restaurant_orders.csv", newline="", encoding="utf-8") as file:
        results = [
            row for row in csv.DictReader(file)
            if row["customer_email"].lower() == email.lower()
        ]

    return results or {"error": f"No orders found for {email}."}


### 4. Describe the functions that OpenAI can call.

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_order",
            "description": "Get the details and current status of a specific restaurant order using its order ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "integer",
                        "description": "The unique order ID."
                    }
                },
                "required": ["order_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_orders_by_customer",
            "description": "Get all restaurant orders belonging to a customer using their email address.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {
                        "type": "string",
                        "description": "The customer's email address."
                    }
                },
                "required": ["email"]
            }
        }
    }
]


### 5. Execute whichever tool OpenAI requests.

In [ ]:
def run_tool(tool_call):
    function_name = tool_call.function.name
    arguments = json.loads(tool_call.function.arguments)

    if function_name == "get_order":
        result = get_order(arguments["order_id"])
    elif function_name == "get_orders_by_customer":
        result = get_orders_by_customer(arguments["email"])
    else:
        result = {"error": f"Unknown tool: {function_name}"}

    return function_name, arguments, result


### 6. Build the information that Gemini will use to evaluate the assistant.

In [ ]:
def build_evaluator_prompt(
    user_message,
    assistant_response,
    tool_calls=None,
    tool_results=None
):
    return f"""
# Evaluator Rules

{evaluator_rules}

# Restaurant Knowledge Base

{restaurant_info}

# User Request

{user_message}

# Assistant Response

{assistant_response}

# Tool Calls

{json.dumps(tool_calls or [], indent=2)}

# Tool Results

{json.dumps(tool_results or [], indent=2)}
"""


### 7. Ask Gemini to evaluate the assistant response and tool activity.

In [ ]:
def evaluate_response(
    user_message,
    assistant_response,
    tool_calls=None,
    tool_results=None
):
    response = gemini_client.chat.completions.create(
        model="gemini-3.5-flash-lite",
        messages=[
            {"role": "system", "content": evaluator_rules},
            {
                "role": "user",
                "content": build_evaluator_prompt(
                    user_message,
                    assistant_response,
                    tool_calls,
                    tool_results
                )
            }
        ],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)


### 8. Keep the OpenAI request in one reusable function.

In [ ]:
def call_openai(messages):
    return openai.chat.completions.create(
        model="gpt-5.4-mini",
        messages=messages,
        tools=tools
    )


### 9. Run the full agent, tool-calling loop, and evaluator feedback loop.

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}]

    for item in history:
        messages.append({
            "role": item["role"],
            "content": item["content"][0]["text"]
        })

    messages.append({"role": "user", "content": message})

    tool_calls_for_evaluator = []
    tool_results_for_evaluator = []

    response = call_openai(messages)

    # Agent loop: OpenAI can request one or more tools.
    while response.choices[0].finish_reason == "tool_calls":
        assistant_message = response.choices[0].message
        messages.append(assistant_message)

        for tool_call in assistant_message.tool_calls:
            function_name, arguments, result = run_tool(tool_call)

            tool_calls_for_evaluator.append({
                "tool": function_name,
                "arguments": arguments
            })

            tool_results_for_evaluator.append({
                "tool": function_name,
                "result": result
            })

            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": json.dumps(result)
            })

        response = call_openai(messages)

    assistant_response = response.choices[0].message.content

    # Gemini evaluates the final response and any tool activity.
    evaluation = evaluate_response(
        message,
        assistant_response,
        tool_calls_for_evaluator,
        tool_results_for_evaluator
    )

    print(f"Evaluator: {evaluation}")

    if not evaluation["passed"]:
        # Feed evaluator feedback back to OpenAI so it can correct itself.
        messages.append({
            "role": "user",
            "content": f"""
The evaluator rejected your previous response.

Reason:
{evaluation["reason"]}

Do not repeat the rejected action.
Provide a safe response to the original user request.
"""
        })

        response = call_openai(messages)
        assistant_response = response.choices[0].message.content

    return assistant_response


### 10. Launch the Gradio chat interface.

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)
